# CELL 1 -- Install Libraries

## What this cell does and why

Before writing any code, we need to install the software tools (called **libraries**) our project depends on.
Think of libraries like apps -- each one does a specific job, and we install them once at the start.

Here is what each library does:

| Library | What it does |
|---|---|
| `ultralytics` | The YOLOv8 framework -- this is the AI model we will train to detect traffic lights |
| `opencv-python-headless` | Reads and writes images and videos (the 'headless' version works in notebooks without a display) |
| `pandas` | Organizes data into tables (like Excel, but in Python) |
| `matplotlib` | Draws charts and plots |
| `seaborn` | Makes prettier charts on top of matplotlib |
| `scikit-learn` | Provides tools for splitting data into train/validation sets |
| `tqdm` | Shows a progress bar so you can see how long a loop will take |
| `PyYAML` | Reads and writes YAML config files -- YOLOv8 needs one to know where your data is |
| `onnx` | Lets us export the trained model to a universal format for deployment on any device |
| `Pillow` | Opens and inspects image files |

**Note:** The `!` at the start of a line tells Jupyter to run a terminal command instead of Python code.

## What to expect after running this cell

A stream of installation messages, then a clean version table. If any row shows `*** NOT FOUND ***`, re-run this cell.

> **No GPU on your machine?** You will see a warning at the bottom. Training will still work but will take 12-20 hours.
> Consider using Google Colab (colab.research.google.com) for a free GPU.

In [1]:
# CELL 1 - Install Libraries

# Install all required packages.
# The -q flag suppresses most output to keep things tidy.
# This may take 2-5 minutes the first time.
!pip install -q ultralytics opencv-python-headless pandas matplotlib seaborn scikit-learn tqdm PyYAML onnx Pillow

# ── Confirm all libraries installed correctly ─────────────────────────────────
import importlib

libs = [
    ("ultralytics",  "ultralytics"),
    ("cv2",          "opencv-python-headless"),
    ("pandas",       "pandas"),
    ("matplotlib",   "matplotlib"),
    ("seaborn",      "seaborn"),
    ("sklearn",      "scikit-learn"),
    ("tqdm",         "tqdm"),
    ("yaml",         "PyYAML"),
    ("onnx",         "onnx"),
    ("PIL",          "Pillow"),
]

print("\n" + "=" * 46)
print(f"  {'Library':<28} {'Version'}")
print("=" * 46)
all_ok = True
for import_name, display_name in libs:
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, "__version__", "installed")
        print(f"  {display_name:<28} {version}")
    except ImportError:
        print(f"  {display_name:<28} *** NOT FOUND -- re-run this cell ***")
        all_ok = False
print("=" * 46)

# GPU Check
import torch
print()
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU  : {gpu_name}")
    print(f"  VRAM : {vram_gb:.1f} GB")
else:
    print("  WARNING: No GPU found.")


print()
if all_ok:
    print("  All libraries installed successfully. Say 'next' to receive Cell 2.")


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

  Library                      Version
  ultralytics                  8.3.222
  opencv-python-headless       4.13.0
  pandas                       2.3.3
  matplotlib                   3.10.7
  seaborn                      0.13.2
  scikit-learn                 1.7.2
  tqdm                         4.67.1
  PyYAML                       6.0.3
  onnx                         1.21.0
  Pillow                       12.0.0

  GPU  : NVIDIA GeForce RTX 5090
  VRAM : 33.7 GB

  All libraries installed successfully. Say 'next' to receive Cell 2.


# CELL 2 -- Import Libraries & Configure Paths

## What this cell does and why

This cell does two important things:

**Part 1 -- Import libraries:** Installing a library (Cell 1) just puts it on your computer.
Importing it is how you actually load it into your current session so your code can use it.
Think of it like installing an app vs. opening it.

**Part 2 -- Define all file paths:** Your dataset lives in specific folders on your hard drive.
Instead of typing the full path every time we need a file, we store each path in a variable
once at the top of the notebook. This way, if anything ever moves, you only need to change one line.

It also **verifies** that your dataset is actually where we expect it to be -- if any folder
is missing or empty, you will see a clear error message here instead of a confusing crash 10 cells later.

**Technical term:** A **path** is just the address of a file or folder on your computer,
like `C:/Users/you/Documents/file.txt` on Windows or `/home/you/file.txt` on Linux.

## What to expect after running this cell

A checklist showing each dataset folder with a file count:
```
[OK] train/img : 20535 images
[OK] train/ann : 20535 annotation files
[OK] test/img  : 22481 images
[OK] test/ann  : 22481 annotation files
```
If any line shows `[MISSING]`, the folder path is wrong -- double-check the `DATASET_ROOT` variable.

In [2]:
# CELL 2 - Import Libraries & Configure Paths

# ── Imports ───────────────────────────────────────────────────────────────────
import json
import os
import re
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import yaml
import torch
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# Make plots look clean
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
random.seed(42)

print("All libraries imported successfully.")
print()

# ── Path configuration ────────────────────────────────────────────────────────
# Change DATASET_ROOT if your dataset is in a different location.
DATASET_ROOT = Path("/home/auto_drive/Desktop/Traffic------new/Dataset")

# Source data (read-only -- we never modify the original dataset)
TRAIN_IMG_DIR = DATASET_ROOT / "train" / "img"
TRAIN_ANN_DIR = DATASET_ROOT / "train" / "ann"
TEST_IMG_DIR  = DATASET_ROOT / "test"  / "img"
TEST_ANN_DIR  = DATASET_ROOT / "test"  / "ann"
META_JSON     = DATASET_ROOT / "meta.json"

# Output folder (will be created in Cell 9 -- do not create it now)
YOLO_ROOT     = Path("/home/auto_drive/Desktop/Traffic------new/yolo_dataset")

# ── Verify dataset is present ─────────────────────────────────────────────────
checks = [
    (TRAIN_IMG_DIR, "train/img",  ".jpg"),
    (TRAIN_ANN_DIR, "train/ann",  ".jpg.json"),
    (TEST_IMG_DIR,  "test/img",   ".jpg"),
    (TEST_ANN_DIR,  "test/ann",   ".jpg.json"),
]

print("Dataset verification:")
print("-" * 45)
all_good = True
for folder, label, ext in checks:
    if folder.exists():
        # Count only the expected file type
        suffix = ext.split('.')[-1]   # 'jpg' or 'json'
        count = len([f for f in folder.iterdir() if f.name.endswith(ext)])
        status = "OK" if count > 0 else "EMPTY"
        print(f"  [{status}] {label:<14} {count:>6} files ({ext})")
        if count == 0:
            all_good = False
    else:
        print(f"  [MISSING] {label} -- folder not found at:")
        print(f"            {folder}")
        all_good = False

print("-" * 45)

if META_JSON.exists():
    meta = json.loads(META_JSON.read_text())
    class_names_raw = [c['title'] for c in meta.get('classes', [])]
    print(f"  [OK] meta.json  -- {len(class_names_raw)} classes found")
else:
    print(f"  [MISSING] meta.json")
    all_good = False

print()
if all_good:
    total = sum(len([f for f in d.iterdir() if f.name.endswith('.jpg')])
                for d in [TRAIN_IMG_DIR, TEST_IMG_DIR])
    print(f"  Dataset ready. Total images: {total:,}  (train + test)")
    print("  Say 'next' to receive Cell 3.")
else:
    print("  One or more folders are missing. Check DATASET_ROOT above.")

All libraries imported successfully.

Dataset verification:
---------------------------------------------
  [OK] train/img       20535 files (.jpg)
  [OK] train/ann       20535 files (.jpg.json)
  [OK] test/img        22481 files (.jpg)
  [OK] test/ann        22481 files (.jpg.json)
---------------------------------------------
  [OK] meta.json  -- 14 classes found

  Dataset ready. Total images: 43,016  (train + test)
  Say 'next' to receive Cell 3.


# CELL 3 -- Understand the Annotation Format

## What this cell does and why

Before we process all 20,535 annotation files, we need to understand what is inside just **one** of them.
This is the most important learning step -- if you understand one file, you understand all of them.

**What is an annotation file?**
An annotation file is a text file that tells the computer:
- Which objects are in a photo (e.g., a green traffic light)
- Exactly where in the photo each object is (a box around it, given as pixel coordinates)

**What format are our annotation files in?**
They are **JSON** files -- a common text format for storing structured data, like a nested dictionary.
JSON stands for JavaScript Object Notation, but you do not need to know JavaScript to use it.

**Important file naming quirk:** Our annotation files have a double extension: `.jpg.json`
(example: `dayClip10--00000.jpg.json`). The matching image is simply `dayClip10--00000.jpg`.

This cell opens one annotation file, prints its raw contents, and then explains each part in plain English.

## What to expect after running this cell

First: the raw JSON text of one annotation file.
Then: a plain-English breakdown of every field, including the pixel coordinates of each bounding box.

In [3]:
# CELL 3 - Understand the Annotation Format

# Pick the first annotation file in the training set
# sorted() puts files in alphabetical order so we always get the same one
ann_files = sorted(TRAIN_ANN_DIR.glob("*.jpg.json"))
sample_ann_path = ann_files[0]

print(f"Annotation file : {sample_ann_path.name}")
print(f"Matching image  : {sample_ann_path.stem}")  # .stem removes the last '.json'
print(f"Image lives at  : {TRAIN_IMG_DIR / sample_ann_path.stem}")
print()

# ── Step 1: Print the raw JSON ────────────────────────────────────────────────
with open(sample_ann_path) as f:
    ann = json.load(f)

print("=" * 55)
print("RAW CONTENT OF THE ANNOTATION FILE (pretty-printed):")
print("=" * 55)
print(json.dumps(ann, indent=2))
print()

# ── Step 2: Break it down field by field ─────────────────────────────────────
print("=" * 55)
print("PLAIN-ENGLISH BREAKDOWN:")
print("=" * 55)

img_w = ann['size']['width']
img_h = ann['size']['height']
print(f"\n[size]")
print(f"  The image is {img_w} pixels wide and {img_h} pixels tall.")
print(f"  Every coordinate in this file is measured in pixels within this canvas.")

print(f"\n[tags]")
for tag in ann.get('tags', []):
    name = tag['name']
    value = tag.get('value', 'yes')
    if name == 'day':
        print(f"  'day' tag present  --> this is a daytime image")
    elif name == 'night':
        print(f"  'night' tag present --> this is a nighttime image")
    elif name == 'sequence':
        print(f"  sequence = '{value}'  --> this frame belongs to clip/sequence '{value}'")
    elif name == 'track frame number':
        print(f"  frame number = {value}  --> this is frame #{value} within the sequence")

print(f"\n[objects]  -- {len(ann['objects'])} traffic light(s) found in this image")
print()

for i, obj in enumerate(ann['objects']):
    pts = obj['points']['exterior']
    # The two corners of the bounding box
    x1 = min(pts[0][0], pts[1][0])
    y1 = min(pts[0][1], pts[1][1])
    x2 = max(pts[0][0], pts[1][0])
    y2 = max(pts[0][1], pts[1][1])
    box_w = x2 - x1
    box_h = y2 - y1

    print(f"  Object #{i+1}")
    print(f"    classTitle : '{obj['classTitle']}'")
    print(f"                 --> the raw label assigned by the dataset creators")
    print(f"    top-left   : ({x1}, {y1}) pixels from the top-left corner of the image")
    print(f"    bot-right  : ({x2}, {y2}) pixels")
    print(f"    box size   : {box_w} px wide  x  {box_h} px tall")
    pct = (box_w * box_h) / (img_w * img_h) * 100
    print(f"    area       : {pct:.3f}% of the full image  (traffic lights are TINY)")
    print()

print("Key takeaway: 'points.exterior' gives us [[x1,y1],[x2,y2]] -- the two opposite")
print("corners of a rectangle around the traffic light. In Cell 10 we will convert")
print("these pixel coordinates into the normalized format that YOLOv8 expects.")

Annotation file : dayClip1--00000.jpg.json
Matching image  : dayClip1--00000.jpg
Image lives at  : /home/auto_drive/Desktop/Traffic------new/Dataset/train/img/dayClip1--00000.jpg

RAW CONTENT OF THE ANNOTATION FILE (pretty-printed):
{
  "description": "",
  "tags": [
    {
      "id": 21476082,
      "tagId": 32241,
      "name": "sequence",
      "value": "Clip1",
      "labelerLogin": "inbox@datasetninja.com",
      "createdAt": "2024-03-10T15:15:43.979Z",
      "updatedAt": "2024-03-10T15:15:43.979Z"
    },
    {
      "id": 21476083,
      "tagId": 32242,
      "name": "day",
      "value": null,
      "labelerLogin": "inbox@datasetninja.com",
      "createdAt": "2024-03-10T15:15:43.979Z",
      "updatedAt": "2024-03-10T15:15:43.979Z"
    },
    {
      "id": 21476084,
      "tagId": 32240,
      "name": "track frame number",
      "value": 0,
      "labelerLogin": "inbox@datasetninja.com",
      "createdAt": "2024-03-10T15:15:43.979Z",
      "updatedAt": "2024-03-10T15:15:43.979Z"